# Body Type Classifier — Colab Training

Fine-tunes a lightweight CNN (MobileNetV3) to classify body shapes from full-body photos.

**Output:** `body_type_model.onnx` (~5 MB) — ready for Flask inference.

In [ ]:
!pip install torch torchvision onnx onnxruntime pillow numpy scikit-learn tqdm

## 1. Dataset

Dataset in `/content/body_dataset/` with subdirectories:
```
body_dataset/
  rectangle/     # img001.jpg ...
  pear/
  apple/
  hourglass/
  inverted_triangle/
```


In [ ]:
import os, torch, torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {DEVICE}")

CLASSES = ["rectangle", "pear", "apple", "hourglass", "inverted_triangle"]
NUM_CLASSES = len(CLASSES)
IMG_SIZE = 224

## 2. Dataset Loader

In [ ]:
class BodyDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        return self.transform(img), self.labels[idx]


def load_dataset(root="/content/body_dataset"):
    paths, labels = [], []
    for label, cls in enumerate(CLASSES):
        dir_path = os.path.join(root, cls)
        if not os.path.isdir(dir_path):
            print(f"WARNING: {dir_path} not found, skipping")
            continue
        for fname in os.listdir(dir_path):
            if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                paths.append(os.path.join(dir_path, fname))
                labels.append(label)
    return paths, labels


paths, labels = load_dataset()
print(f"Found {len(paths)} images")
if len(paths) == 0:
    print("No dataset found. Run the next cell to generate synthetic data.")

## 3. (Optional) Generate Synthetic Dataset

If you don't have a labeled dataset, this cell creates one from random augmentations.

In [ ]:
def generate_synthetic(output_dir="/content/body_dataset", samples_per_class=50):
    """Generate a toy dataset using solid color rectangles as "body shapes".
    
    Real usage: replace this with actual body images from:
    - 3D-Avatars dataset
    - Human3.6M (pose only, not shape)
    - Any fashion model dataset where you can manually label
    """
    import random
    os.makedirs(output_dir, exist_ok=True)
    for cls in CLASSES:
        os.makedirs(f"{output_dir}/{cls}", exist_ok=True)
    
    # Simulated shapes using ellipses (shoulder/hip ratios)
    ratios = {
        "rectangle":      (1.0, 0.85),  # shoulder_w, hip_w
        "pear":           (0.8, 1.1),
        "apple":          (0.95, 0.95),
        "hourglass":      (1.0, 1.0),
        "inverted_triangle": (1.2, 0.85),
    }
    
    for cls in CLASSES:
        sw, hw = ratios[cls]
        for i in range(samples_per_class):
            # Add variation
            sw_var = sw + random.uniform(-0.08, 0.08)
            hw_var = hw + random.uniform(-0.08, 0.08)
            img = np.ones((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8) * 240
            cy, cx = IMG_SIZE // 2, IMG_SIZE // 2
            shoulder_w = int(IMG_SIZE * 0.3 * sw_var)
            hip_w = int(IMG_SIZE * 0.3 * hw_var)
            body_h = int(IMG_SIZE * 0.6)
            # Draw shoulders (ellipse)
            cv2.ellipse(img, (cx, cy - body_h//3), (shoulder_w, 15), 0, 0, 360, (200, 180, 160), -1)
            # Draw hips (ellipse)
            cv2.ellipse(img, (cx, cy + body_h//3), (hip_w, 15), 0, 0, 360, (180, 160, 140), -1)
            # Connect with body polygon
            pts = np.array([
                [cx - shoulder_w, cy - body_h//3 + 10],
                [cx + shoulder_w, cy - body_h//3 + 10],
                [cx + hip_w, cy + body_h//3 - 10],
                [cx - hip_w, cy + body_h//3 - 10],
            ], np.int32)
            cv2.fillPoly(img, [pts], (190, 170, 150))
            cv2.imwrite(f"{output_dir}/{cls}/syn_{i:04d}.png", img)
    print(f"Generated {samples_per_class * len(CLASSES)} synthetic images")

if len(paths) == 0:
    import cv2
    generate_synthetic()
    paths, labels = load_dataset()
    print(f"Now using {len(paths)} synthetic images")

## 4. Training

In [ ]:
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_paths, val_paths, train_labels, val_labels = train_test_split(
    paths, labels, test_size=0.2, random_state=42, stratify=labels
)

train_ds = BodyDataset(train_paths, train_labels, transform)
val_ds = BodyDataset(val_paths, val_labels, val_transform)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, NUM_CLASSES)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

print(f"Train: {len(train_ds)} samples, Val: {len(val_ds)} samples")

In [ ]:
EPOCHS = 30
best_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for images, target in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, target = images.to(DEVICE), target.to(DEVICE)
        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    scheduler.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, target in val_loader:
            images, target = images.to(DEVICE), target.to(DEVICE)
            output = model(images)
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()
    acc = 100.0 * correct / total
    print(f"Loss: {train_loss/len(train_loader):.4f} | Val Acc: {acc:.2f}%")

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "/content/best_body_type.pth")

print(f"Best validation accuracy: {best_acc:.2f}%")

## 5. Export to ONNX

In [ ]:
model.load_state_dict(torch.load("/content/best_body_type.pth"))
model.eval()

dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
torch.onnx.export(
    model, dummy_input, "/content/body_type_model.onnx",
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
    opset_version=17,
)
print("Model exported to /content/body_type_model.onnx")

import os
size_mb = os.path.getsize("/content/body_type_model.onnx") / 1e6
print(f"Model size: {size_mb:.1f} MB")

## 6. Test Inference

Test the exported model on a sample image.

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession("/content/body_type_model.onnx")
input_name = session.get_inputs()[0].name

def predict(image_path):
    img = Image.open(image_path).convert("RGB")
    img = val_transform(img).unsqueeze(0).numpy()
    outputs = session.run(None, {input_name: img})
    pred_idx = outputs[0][0].argmax()
    return CLASSES[pred_idx]

# Test with first validation image
if len(val_paths) > 0:
    pred = predict(val_paths[0])
    true = CLASSES[val_labels[0]]
    print(f"Predicted: {pred}, True: {true}")

## 7. Download the Model

Copy `body_type_model.onnx` to your Flask server's directory.

In [ ]:
from google.colab import files
files.download("/content/body_type_model.onnx")

---

## Deployment

Place `body_type_model.onnx` in `server/` and add inference to `server/body_type.py`:

```python
import onnxruntime as ort

_session = ort.InferenceSession("body_type_model.onnx")
_input_name = _session.get_inputs()[0].name

def classify_with_onnx(img_rgb: np.ndarray) -> str:
    img = cv2.resize(img_rgb, (224, 224))
    img = img.astype(np.float32) / 255.0
    img = (img - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]
    img = img.transpose(2, 0, 1)[np.newaxis, ...]
    outputs = _session.run(None, {_input_name: img})
    idx = outputs[0][0].argmax()
    return ["rectangle", "pear", "apple", "hourglass", "inverted_triangle"][idx]
```

The running server (in `body_type.py`) already falls back to MediaPipe Pose if ONNX model not found.